In [1]:
%run shared_imports.py
from decimal import Decimal
from dataclasses import dataclass
from sqlalchemy.orm.query import Query
import gspread
from gspread_dataframe import get_as_dataframe, set_with_dataframe


In [2]:
engine = make_engine("settings.toml")
session = Session(engine)

In [15]:
win_count = 0
loss_count = 0

query = session.query(Feedback).join(Round).filter(Round.start_datetime >= datetime(2024, 5, 6))

with Session(engine) as session:
    for feedback in query.all():
        if feedback.key_name.startswith("Blob survives"):
            win_count += 1
        elif feedback.key_name.startswith("Blob nuclear victories"):
            win_count += 1
        elif feedback.key_name.startswith("Blob dies"):
            loss_count += 1
        elif feedback.key_name == "biohazards":
            for fate, antags in feedback.items():
                if fate == "survived" and "Blob" in antags:
                    win_count += antags["Blob"]
                elif fate == "defeated" and "Blob" in antags:
                    loss_count += antags["Blob"]


In [16]:
win_count

41

In [17]:
loss_count

44

In [7]:
%debug

> c:\users\warriorstar\appdata\local\temp\ipykernel_8584\1657116451.py(20)<module>()



ipdb>  fate


'defeated'


ipdb>  antags


{'Blob': 1}


ipdb>  q


In [3]:
biohazards

{'Blob': defaultdict(int,
             {'Blob survives to normal round en': 24,
              'Blob nuclear victories': 16,
              'Blob dies normal end': 37,
              'Blob survives to admin round end': 1,
              'Blob dies station nuked': 4})}

In [4]:
session = Session(engine)
feedbacks = session.query(Feedback).join(Round).filter(Feedback.key_name=='vampire_subclasses', Round.start_datetime >= datetime.datetime(2024, 1, 1)).order_by(desc(Round.start_datetime)).all()

In [8]:
classtypes = defaultdict(int)
for feedback in session.query(Feedback).join(Round).filter(
        Feedback.key_name=='vampire_subclasses',
        Round.start_datetime >= datetime.datetime(2024, 1, 1)).all():
    for subclass, count in feedback.items():
        classtypes[subclass] += count

In [9]:
classtypes

defaultdict(int,
            {'gargantua': 408,
             'umbrae': 345,
             'dantalion': 274,
             'hemomancer': 418,
             'ancient': 5})

In [13]:
%debug

> c:\users\warriorstar\appdata\local\temp\ipykernel_42872\3624506251.py(14)<module>()



ipdb>  biohazard_type


'Princess Terrors'


ipdb>  biohazards


<class 'dict'>


ipdb>  type(biohazard_type)


<class 'str'>


ipdb>  type(biohazards)


<class 'type'>


ipdb>  q


In [15]:
r.start_datetime > datetime.datetime(2023, 1, 17)

True

In [5]:
%debug

> d:\externalrepos\ss13_blackbox_tools\model.py(235)__getitem__()
    233 
    234     def __getitem__(self, key):
--> 235         return self.json['data'].__getitem__(key)
    236 
    237     def keys(self):



ipdb>  key


0


ipdb>  up


> c:\users\warriorstar\appdata\local\temp\ipykernel_24884\2359386715.py(13)<module>()



...

ipdb>  votes.data


*** AttributeError: 'Feedback' object has no attribute 'data'


ipdb>  votes.keys()


dict_keys(['crew transfer', 'map'])


ipdb>  q


In [17]:
vote_counts

defaultdict(int,
            {'NSS Cyberiad (Cyberiad)': 14608,
             'NSS Cerebron (MetaStation)': 3971,
             'NSS Kerberos (Delta)': 4778,
             'NSS Farragus (CereStation)': 5170})

In [13]:
total_count

1354

In [8]:
for name, count in vote_counts.items():
    print(f"{name},{count}")

NSS Cyberiad (Cyberiad),14608
NSS Cerebron (MetaStation),3971
NSS Kerberos (Delta),4778
NSS Farragus (CereStation),5170


In [22]:
r.map_name

'MetaStation'

In [28]:
from collections import defaultdict
from datetime import datetime
import json

r_count = 0
total_count = 0

gamemode_counts = dict()
vote_counts = defaultdict(int)
earliest_map = dict()
earliest_map_id = dict()

items = defaultdict(int)
with Session(engine) as session:
    for r in session.query(Round).all():
        if r.map_name in earliest_map:
            if r.start_datetime < earliest_map[r.map_name]:
                earliest_map_id[r.map_name] = r.id
                earliest_map[r.map_name] = r.start_datetime
        else:
            earliest_map_id[r.map_name] = r.id
            earliest_map[r.map_name] = r.start_datetime
        # if r.start_datetime >= datetime.datetime(2023, 1, 17):
            # gamemode_counts.setdefault(r.map_name, defaultdict(int))
            # gamemode_counts[r.map_name][r.game_mode] += 1
        # if r.has_feedback("votes"):
        #     votes = r.feedback("votes")
        #     if isinstance(votes.json, str):
        #         data = json.loads(votes.json)
        #     else:
        #         data = votes.json
        #     print(data['data'])
        #     if "map" in data['data']:
                
        #         total_count += 1
        #         for name, count in data['data']["map"].items():
        #             vote_counts[name] += count



In [29]:
earliest_map

{'Cyberiad': datetime.datetime(2023, 7, 24, 8, 58, 46),
 'MetaStation': datetime.datetime(2023, 7, 25, 4, 14, 30),
 'CereStation': datetime.datetime(2023, 7, 25, 11, 7, 21),
 'Delta': datetime.datetime(2023, 7, 25, 12, 36, 50)}

In [30]:
earliest_map_id

{'Cyberiad': 36657, 'MetaStation': 36667, 'CereStation': 36670, 'Delta': 36671}

In [23]:
vote_counts

defaultdict(int,
            {'NSS Cyberiad (Cyberiad)': 16582,
             'NSS Cerebron (MetaStation)': 4625,
             'NSS Kerberos (Delta)': 5335,
             'NSS Farragus (CereStation)': 5703})

In [37]:
map_counts = defaultdict(int)
for map, counts in gamemode_counts.items():
    map_counts[map] = sum(counts.values())
    
for map, counts in gamemode_counts.items():
    for gamemode, count in counts.items():
        avg = count / map_counts[map]
        print(f"{map}: {gamemode} {avg}")

Cyberiad: AutoTraitor 0.09000989119683482
Cyberiad: nuclear emergency 0.049950544015825916
Cyberiad: traitor_changeling 0.10187932739861523
Cyberiad: cult 0.057368941641938676
Cyberiad: extended 0.1261127596439169
Cyberiad: traitor_vampire 0.10781404549950543
Cyberiad: vampire 0.11275964391691394
Cyberiad: wizard 0.0791295746785361
Cyberiad: traitor 0.06973293768545995
Cyberiad: changeling 0.09594460929772503
Cyberiad: Trifecta 0.09149357072205737
Cyberiad: traitor+changeling 0.002967359050445104
Cyberiad: traitor+vampire 0.0034619188921859545
Cyberiad: None 0.0004945598417408506
Cyberiad: revolution 0.005440158259149357
Cyberiad: ragin' mages 0.004945598417408506
Cyberiad: abduction 0.0004945598417408506
MetaStation: changeling 0.11403508771929824
MetaStation: extended 0.12280701754385964
MetaStation: Trifecta 0.05555555555555555
MetaStation: cult 0.05555555555555555
MetaStation: traitor_changeling 0.10526315789473684
MetaStation: nuclear emergency 0.04678362573099415
MetaStation: tra

In [35]:
sum(gamemode_counts['Cyberiad'].values())

2022

In [36]:
182/2022

0.09000989119683482

In [31]:
map_counts

defaultdict(int,
            {'Cyberiad': 2022,
             'MetaStation': 342,
             'CereStation': 298,
             'Delta': 495})

In [23]:
dict.setdefault

<method 'setdefault' of 'dict' objects>